# Sherlock Holmes Next-Sentence Transformer

### A transparent, from-scratch language-model experiment

**Project:** Alejandro Reynoso  
**Objective:** train a small causal Transformer on public-domain Sherlock Holmes texts and use it to generate the sentence that might come next.

> “Data! Data! Data! I can’t make bricks without clay.”

This notebook deliberately avoids a pretrained language model. We will build the vocabulary, the training examples, the attention model, and the generation loop ourselves. The result is not meant to rival a commercial LLM; it is meant to make the full mechanism visible.

---

## What the notebook does

1. Downloads five Sherlock Holmes works from Project Gutenberg.
2. Removes Gutenberg headers and footers and segments the prose into sentences.
3. Builds a simple word-and-punctuation tokenizer from the corpus.
4. Converts consecutive sentences into one causal token stream.
5. Trains a compact Transformer to predict the **next token** at every position.
6. Uses those token predictions to generate the **next complete sentence**.
7. Reports validation loss and perplexity, then saves the model and vocabulary.

### An important distinction

The phrase *next sentence prediction* can mean two different things:

- **BERT-style NSP:** a binary classifier asks whether sentence B really follows sentence A.
- **Generative continuation:** a causal language model writes a plausible sentence after the prompt.

This notebook implements the second, more useful interpretation. Internally, it learns one token at a time; externally, it stops when a complete sentence has been produced.

### Runtime

In Colab, choose **Runtime → Change runtime type → T4 GPU**. The default “quick” configuration is designed for a pedagogical run. A larger configuration is included for a stronger model.

## 1. Imports, reproducibility, and experiment controls

No specialized NLP framework is required. PyTorch supplies the Transformer components; the tokenizer is implemented below with regular expressions so every step remains inspectable.

In [ ]:
import json
import hashlib
import math
import random
import re
import time
import urllib.request
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
else:
    print("Tip: enable a T4 GPU in Colab for much faster training.")

In [ ]:
# Two presets:
# - "quick": a compact classroom experiment, usually a few minutes on a T4.
# - "stronger": more context, capacity, data, and training time.
RUN_MODE = "quick"  # change to "stronger" for a better model

@dataclass
class Config:
    block_size: int
    stride: int
    d_model: int
    n_heads: int
    n_layers: int
    d_ff: int
    dropout: float
    batch_size: int
    epochs: int
    learning_rate: float
    max_tokens: int | None
    max_vocab: int
    min_frequency: int

if RUN_MODE == "quick":
    CFG = Config(
        block_size=96,
        stride=64,
        d_model=192,
        n_heads=4,
        n_layers=4,
        d_ff=768,
        dropout=0.15,
        batch_size=64 if DEVICE.type == "cuda" else 24,
        epochs=4 if DEVICE.type == "cuda" else 2,
        learning_rate=3e-4,
        max_tokens=400_000,
        max_vocab=16_000,
        min_frequency=2,
    )
else:
    CFG = Config(
        block_size=160,
        stride=96,
        d_model=256,
        n_heads=8,
        n_layers=6,
        d_ff=1024,
        dropout=0.15,
        batch_size=48 if DEVICE.type == "cuda" else 16,
        epochs=8 if DEVICE.type == "cuda" else 3,
        learning_rate=2.5e-4,
        max_tokens=None,
        max_vocab=24_000,
        min_frequency=1,
    )

CFG

## 2. Obtain the public-domain corpus

The corpus uses five works by Arthur Conan Doyle:

- *The Adventures of Sherlock Holmes* — Project Gutenberg #1661
- *The Return of Sherlock Holmes* — #108
- *The Hound of the Baskervilles* — #2852
- *The Sign of the Four* — #2097
- *A Study in Scarlet* — #244

Project Gutenberg marks these editions as public domain in the United States. Its license and distribution terms still apply. If you run this notebook elsewhere, confirm the copyright status in your jurisdiction.

In [ ]:
BOOKS = {
    1661: "The Adventures of Sherlock Holmes",
    108: "The Return of Sherlock Holmes",
    2852: "The Hound of the Baskervilles",
    2097: "The Sign of the Four",
    244: "A Study in Scarlet",
}

DATA_DIR = Path("sherlock_data")
DATA_DIR.mkdir(exist_ok=True)

def download_gutenberg(book_id: int) -> str:
    # Download a UTF-8 Project Gutenberg plain-text file, with a local cache.
    destination = DATA_DIR / f"pg{book_id}.txt"
    if destination.exists():
        return destination.read_text(encoding="utf-8")

    candidate_urls = [
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
        f"https://www.gutenberg.org/ebooks/{book_id}.txt.utf-8",
    ]
    last_error = None
    for url in candidate_urls:
        try:
            request = urllib.request.Request(
                url, headers={"User-Agent": "Educational Colab notebook"}
            )
            with urllib.request.urlopen(request, timeout=45) as response:
                raw = response.read()
            text = raw.decode("utf-8-sig", errors="replace")
            destination.write_text(text, encoding="utf-8")
            return text
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Could not download Gutenberg book {book_id}") from last_error

raw_books = {}
corpus_manifest = {}
for book_id, title in BOOKS.items():
    raw_books[book_id] = download_gutenberg(book_id)
    corpus_manifest[book_id] = {
        "title": title,
        "source": f"https://www.gutenberg.org/ebooks/{book_id}",
        "sha256": hashlib.sha256(
            raw_books[book_id].encode("utf-8")
        ).hexdigest(),
    }
    print(f"{book_id:>4} | {len(raw_books[book_id]):>9,} characters | {title}")

## 3. Clean the books and identify sentences

The cleaning is intentionally conservative:

- Gutenberg metadata outside the `START` and `END` markers is removed.
- Curly quotation marks and long dashes are normalized.
- line-wrapped prose is joined into paragraphs.
- obvious illustration notes and repeated whitespace are removed.

Sentence segmentation is rule-based. This keeps the notebook self-contained, though it will occasionally treat abbreviations imperfectly.

In [ ]:
def strip_gutenberg_boilerplate(text: str) -> str:
    start = re.search(
        r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    end = re.search(
        r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if start:
        text = text[start.end():]
    if end:
        text = text[:end.start()]
    return text

def clean_book(text: str) -> str:
    text = strip_gutenberg_boilerplate(text)
    replacements = {
        "\ufeff": "",
        "\r": "\n",
        "“": '"',
        "”": '"',
        "‘": "'",
        "’": "'",
        "—": " -- ",
        "–": " - ",
        "\xa0": " ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"\[(?:Illustration|Transcriber's Note).*?\]", " ", text,
                  flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"-\n(?=[a-z])", "", text)       # repair line-break hyphenation
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)   # join wrapped lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

ABBREVIATIONS = {
    "mr.", "mrs.", "ms.", "dr.", "st.", "prof.", "sr.", "jr.",
    "etc.", "e.g.", "i.e.", "vs.", "no.", "fig.", "vol.",
}

def split_sentences(text: str) -> list[str]:
    # Start with a punctuation-plus-whitespace boundary.
    rough = re.split(r'(?<=[.!?])\s+(?=(?:["\']?[A-Z]))', text)
    sentences = []
    pending = ""
    for piece in rough:
        piece = re.sub(r"\s+", " ", piece).strip()
        if not piece:
            continue
        piece = f"{pending} {piece}".strip() if pending else piece
        final_word = piece.lower().split()[-1] if piece.split() else ""
        # If a fragment ends in an abbreviation, hold it for the next fragment.
        if final_word in ABBREVIATIONS:
            pending = piece
            continue
        pending = ""
        if 3 <= len(piece.split()) <= 120:
            sentences.append(piece)
    if pending and 3 <= len(pending.split()) <= 120:
        sentences.append(pending)
    return sentences

clean_books = {book_id: clean_book(text) for book_id, text in raw_books.items()}
sentences_by_book = {
    book_id: split_sentences(text) for book_id, text in clean_books.items()
}

for book_id, title in BOOKS.items():
    print(f"{len(sentences_by_book[book_id]):>6,} sentences | {title}")

all_sentences = [
    sentence
    for book_id in BOOKS
    for sentence in sentences_by_book[book_id]
]
print(f"\nTotal: {len(all_sentences):,} sentences")

In [ ]:
print("A few cleaned training sentences:\n")
for sentence in all_sentences[100:106]:
    print("•", sentence)

## 4. Build a transparent word-level tokenizer

The tokenizer separates words from punctuation. Each sentence is enclosed by two structural tokens:

- `<bos>` means “beginning of sentence”
- `<eos>` means “end of sentence”

Other special tokens are `<pad>` and `<unk>`. The latter absorbs rare or unseen words. The model will therefore see a stream such as:

`<bos> I looked at Holmes . <eos> <bos> He smiled quietly . <eos>`

That structure is what allows the generation function to stop after one new sentence.

In [ ]:
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*|[^\w\s]")
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]

def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text)

token_counts = Counter()
for sentence in all_sentences:
    token_counts.update(tokenize(sentence))

kept_tokens = [
    token for token, count in token_counts.most_common(CFG.max_vocab)
    if count >= CFG.min_frequency and token not in SPECIAL_TOKENS
]
itos = SPECIAL_TOKENS + kept_tokens
stoi = {token: index for index, token in enumerate(itos)}

PAD_ID = stoi["<pad>"]
UNK_ID = stoi["<unk>"]
BOS_ID = stoi["<bos>"]
EOS_ID = stoi["<eos>"]

print(f"Raw unique tokens: {len(token_counts):,}")
print(f"Model vocabulary:  {len(itos):,}")
print("Most common tokens:", token_counts.most_common(20))

In [ ]:
def encode_sentence(sentence: str) -> list[int]:
    ids = [stoi.get(token, UNK_ID) for token in tokenize(sentence)]
    return [BOS_ID] + ids + [EOS_ID]

def detokenize(tokens: list[str]) -> str:
    # Reconstruct readable text from word-and-punctuation tokens.
    text = " ".join(token for token in tokens if token not in SPECIAL_TOKENS)
    text = re.sub(r"\s+([,.;:!?%\)\]\}])", r"\1", text)
    text = re.sub(r"([\(\[\{])\s+", r"\1", text)
    text = re.sub(r'"\s+([^"]+?)\s+"', r'"\1"', text)
    text = text.replace(" n't", "n't").replace(" 's", "'s")
    return text.strip()

example = all_sentences[250]
example_ids = encode_sentence(example)
print("Original:    ", example)
print("Tokenized:   ", tokenize(example)[:30])
print("IDs:         ", example_ids[:30])
print("Reconstructed", detokenize([itos[i] for i in example_ids]))

## 5. Encode the corpus and create causal training windows

We preserve sentence order within each book. For every work, the first 90% becomes training data and the final 10% is held out for validation. The per-book split ensures that all five works contribute to both sets.

For a window

`[<bos>, Holmes, looked, ...]`

the target is the same sequence shifted left:

`[Holmes, looked, ..., <eos>]`

A causal attention mask prevents the model from looking to the right. Thus every target must be inferred only from earlier tokens.

In [ ]:
train_pieces = []
valid_pieces = []
per_book_cap = (
    CFG.max_tokens // len(BOOKS) if CFG.max_tokens is not None else None
)

for book_id in BOOKS:
    book_stream = []
    for sentence in sentences_by_book[book_id]:
        book_stream.extend(encode_sentence(sentence))
    if per_book_cap is not None:
        book_stream = book_stream[:per_book_cap]
    split_index = int(0.90 * len(book_stream))
    train_pieces.append(torch.tensor(book_stream[:split_index], dtype=torch.long))
    valid_pieces.append(torch.tensor(book_stream[split_index:], dtype=torch.long))

train_tokens = torch.cat(train_pieces)
valid_tokens = torch.cat(valid_pieces)
token_stream = torch.cat([train_tokens, valid_tokens])

print(f"Total encoded tokens: {len(token_stream):,}")
print(f"Training tokens:      {len(train_tokens):,}")
print(f"Validation tokens:    {len(valid_tokens):,}")

In [ ]:
class TokenWindowDataset(Dataset):
    def __init__(self, tokens: torch.Tensor, block_size: int, stride: int):
        self.tokens = tokens
        self.block_size = block_size
        self.starts = list(range(0, len(tokens) - block_size, stride))

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, index):
        start = self.starts[index]
        chunk = self.tokens[start:start + self.block_size + 1]
        return chunk[:-1], chunk[1:]

train_ds = TokenWindowDataset(train_tokens, CFG.block_size, CFG.stride)
valid_ds = TokenWindowDataset(valid_tokens, CFG.block_size, CFG.block_size)

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=(DEVICE.type == "cuda"),
    generator=generator,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=(DEVICE.type == "cuda"),
)

x_batch, y_batch = next(iter(train_loader))
print(f"Training windows: {len(train_ds):,}")
print(f"Validation windows: {len(valid_ds):,}")
print(f"Batch x shape: {tuple(x_batch.shape)}")
print(f"Batch y shape: {tuple(y_batch.shape)}")

## 6. Define the tiny Transformer

The model has four conceptual stages:

1. **Token embedding** — turns each token ID into a vector.
2. **Position embedding** — tells the model where each token occurs in the window.
3. **Causal Transformer stack** — mixes information through masked self-attention.
4. **Language-model head** — maps each contextual vector to scores over the vocabulary.

The output head shares its weights with the token embedding. This both reduces parameters and ties the input and output representations together.

In [ ]:
class TinySherlockTransformer(nn.Module):
    def __init__(self, vocab_size: int, cfg: Config):
        super().__init__()
        self.block_size = cfg.block_size
        self.d_model = cfg.d_model
        self.token_embedding = nn.Embedding(vocab_size, cfg.d_model)
        self.position_embedding = nn.Embedding(cfg.block_size, cfg.d_model)
        self.embedding_dropout = nn.Dropout(cfg.dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.d_ff,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=cfg.n_layers,
            norm=nn.LayerNorm(cfg.d_model),
            enable_nested_tensor=False,
        )
        self.lm_head = nn.Linear(cfg.d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length = token_ids.shape
        if sequence_length > self.block_size:
            raise ValueError("Input exceeds the configured context window.")

        positions = torch.arange(sequence_length, device=token_ids.device)
        x = self.token_embedding(token_ids) * math.sqrt(self.d_model)
        x = x + self.position_embedding(positions)[None, :, :]
        x = self.embedding_dropout(x)

        # True entries are hidden: token t cannot attend to any future token.
        causal_mask = torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                device=token_ids.device,
                dtype=torch.bool,
            ),
            diagonal=1,
        )
        x = self.transformer(x, mask=causal_mask)
        return self.lm_head(x)

model = TinySherlockTransformer(len(itos), CFG).to(DEVICE)
parameter_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"\nParameters: {parameter_count:,}")
print(f"Trainable:  {trainable_count:,}")

with torch.no_grad():
    test_logits = model(x_batch[:2].to(DEVICE))
print("Output shape:", tuple(test_logits.shape))
assert test_logits.shape == (2, CFG.block_size, len(itos))

## 7. Train the model

Cross-entropy asks the model to place high probability on the true next token. We use:

- AdamW optimization
- gradient clipping for stability
- cosine learning-rate decay
- automatic mixed precision on CUDA
- validation after each epoch

The best validation checkpoint is retained in memory.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    betas=(0.9, 0.95),
    weight_decay=0.1,
)
total_steps = max(1, CFG.epochs * len(train_loader))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=CFG.learning_rate * 0.1
)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for inputs, targets in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(inputs)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        token_count = targets.numel()
        total_loss += loss.item() * token_count
        total_tokens += token_count
    return total_loss / total_tokens

history = {"train_loss": [], "valid_loss": [], "learning_rate": []}
best_valid_loss = float("inf")
best_state = None

for epoch in range(1, CFG.epochs + 1):
    model.train()
    running_loss = 0.0
    seen_tokens = 0
    epoch_start = time.time()

    for step, (inputs, targets) in enumerate(train_loader, start=1):
        inputs = inputs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(inputs)
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        token_count = targets.numel()
        running_loss += loss.item() * token_count
        seen_tokens += token_count

        if step % max(1, len(train_loader) // 4) == 0:
            print(
                f"Epoch {epoch}/{CFG.epochs} | "
                f"step {step:>4}/{len(train_loader)} | "
                f"loss {running_loss / seen_tokens:.4f}"
            )

    train_loss = running_loss / seen_tokens
    valid_loss = evaluate(model, valid_loader)
    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)
    history["learning_rate"].append(optimizer.param_groups[0]["lr"])

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

    elapsed = time.time() - epoch_start
    perplexity = math.exp(min(valid_loss, 20))
    print(
        f"Completed epoch {epoch}: "
        f"train={train_loss:.4f}, valid={valid_loss:.4f}, "
        f"perplexity={perplexity:.2f}, time={elapsed:.1f}s\n"
    )

model.load_state_dict(best_state)
model.to(DEVICE)
print(f"Best validation loss: {best_valid_loss:.4f}")
print(f"Best validation perplexity: {math.exp(min(best_valid_loss, 20)):.2f}")

## 8. Diagnose learning

Loss measures average surprise about the correct next token. Perplexity is its exponential form: lower is better, although it should only be compared across models using the same tokenizer and data.

A widening train–validation gap indicates overfitting. If both losses remain high, the model is undertrained or underpowered.

In [ ]:
epochs_axis = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(8, 4.5))
plt.plot(epochs_axis, history["train_loss"], marker="o", label="Training")
plt.plot(epochs_axis, history["valid_loss"], marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Sherlock Transformer learning curve")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 9. Generate the next sentence

Generation uses four controls:

- **temperature**: lower values are safer and more repetitive; higher values are more surprising.
- **top-k**: keeps only the *k* most likely candidates.
- **top-p**: keeps the smallest candidate set whose cumulative probability reaches *p*.
- **repetition penalty**: slightly discourages recently used tokens.

The function appends `<eos><bos>` after the prompt and stops at the model's next `<eos>`. This explicitly asks for the sentence *after* the supplied text.

In [ ]:
def encode_prompt(prompt: str) -> list[int]:
    prompt_ids = [stoi.get(token, UNK_ID) for token in tokenize(prompt)]
    # Mark the supplied prompt as complete and request a fresh sentence.
    return prompt_ids + [EOS_ID, BOS_ID]

def filter_logits(logits, top_k=40, top_p=0.92):
    logits = logits.clone()
    if top_k is not None and top_k > 0:
        k = min(top_k, logits.numel())
        threshold = torch.topk(logits, k).values[-1]
        logits[logits < threshold] = -float("inf")

    if top_p is not None and 0 < top_p < 1:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        remove = cumulative > top_p
        remove[1:] = remove[:-1].clone()
        remove[0] = False
        logits[sorted_indices[remove]] = -float("inf")
    return logits

@torch.inference_mode()
def predict_next_sentence(
    prompt: str,
    max_new_tokens: int = 55,
    min_new_tokens: int = 5,
    temperature: float = 0.85,
    top_k: int = 40,
    top_p: float = 0.92,
    repetition_penalty: float = 1.12,
    seed: int | None = None,
) -> str:
    if seed is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    model.eval()
    context = encode_prompt(prompt)
    generated = []
    forbidden = {PAD_ID, UNK_ID, BOS_ID}

    for _ in range(max_new_tokens):
        model_input = torch.tensor(
            [context[-CFG.block_size:]], dtype=torch.long, device=DEVICE
        )
        logits = model(model_input)[0, -1] / max(temperature, 1e-5)

        # Do not generate structural tokens in the middle of the sentence.
        for token_id in forbidden:
            logits[token_id] = -float("inf")
        if len(generated) < min_new_tokens:
            logits[EOS_ID] = -float("inf")

        for token_id in set(generated[-20:]):
            if logits[token_id] > 0:
                logits[token_id] /= repetition_penalty
            else:
                logits[token_id] *= repetition_penalty

        logits = filter_logits(logits, top_k=top_k, top_p=top_p)
        probabilities = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probabilities, num_samples=1).item()

        if next_id == EOS_ID:
            break
        generated.append(next_id)
        context.append(next_id)

    words = [itos[token_id] for token_id in generated]
    sentence = detokenize(words)
    if sentence and sentence[-1] not in ".!?":
        sentence += "."
    return sentence

In [ ]:
demo_prompts = [
    "Holmes examined the strange mark upon the window.",
    "The night was cold, and a heavy fog covered Baker Street.",
    "I handed the mysterious letter to my companion.",
    "There was only one clue left in the silent room.",
]

for index, prompt in enumerate(demo_prompts):
    continuation = predict_next_sentence(prompt, seed=100 + index)
    print(f"PROMPT: {prompt}")
    print(f"NEXT:   {continuation}\n")

### Try your own prompt

For a more deterministic answer, try `temperature=0.65`. For a more adventurous answer, try `temperature=1.0`. Because sampling is stochastic, a different seed can produce a different continuation.

In [ ]:
my_prompt = "Sherlock Holmes looked across the table and became suddenly silent."

for sample_seed in [7, 19, 221]:
    print(
        f"Seed {sample_seed:>3}:",
        predict_next_sentence(
            my_prompt,
            temperature=0.82,
            top_k=40,
            top_p=0.92,
            seed=sample_seed,
        ),
    )

## 10. Inspect the model’s immediate beliefs

The next cell does not generate a full sentence. It displays the highest-probability first tokens after the prompt, making the model’s distribution visible.

In [ ]:
@torch.inference_mode()
def show_top_next_tokens(prompt: str, n=15):
    model.eval()
    context = encode_prompt(prompt)
    x = torch.tensor(
        [context[-CFG.block_size:]], dtype=torch.long, device=DEVICE
    )
    probabilities = torch.softmax(model(x)[0, -1], dim=-1)
    values, indices = torch.topk(probabilities, k=n)
    rows = [(itos[i.item()], p.item()) for p, i in zip(values, indices)]
    return rows

for token, probability in show_top_next_tokens(my_prompt):
    print(f"{token!r:>16}  {probability:8.4%}")

## 11. Save the trained artifact

The export contains:

- `model.pt` — learned parameters
- `vocabulary.json` — token-to-ID and ID-to-token mappings
- `config.json` — architecture and experiment configuration
- `training_history.json` — losses and learning-rate history

The ZIP can be downloaded directly from Colab or copied to Google Drive.

In [ ]:
ARTIFACT_DIR = Path("sherlock_transformer_artifact")
ARTIFACT_DIR.mkdir(exist_ok=True)

torch.save(
    {
        "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "config": asdict(CFG),
        "vocab_size": len(itos),
        "best_validation_loss": best_valid_loss,
    },
    ARTIFACT_DIR / "model.pt",
)
(ARTIFACT_DIR / "vocabulary.json").write_text(
    json.dumps({"itos": itos, "stoi": stoi}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(ARTIFACT_DIR / "config.json").write_text(
    json.dumps(
        {
            "run_mode": RUN_MODE,
            "config": asdict(CFG),
            "books": BOOKS,
            "corpus_manifest": corpus_manifest,
            "seed": SEED,
            "tokenizer_pattern": TOKEN_PATTERN.pattern,
        },
        indent=2,
    ),
    encoding="utf-8",
)
(ARTIFACT_DIR / "training_history.json").write_text(
    json.dumps(history, indent=2), encoding="utf-8"
)

ZIP_PATH = Path("sherlock_transformer_artifact.zip")
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in ARTIFACT_DIR.iterdir():
        archive.write(path, arcname=path.name)

print(f"Saved: {ZIP_PATH.resolve()}")
print(f"Size:  {ZIP_PATH.stat().st_size / 1_000_000:.2f} MB")

# In Colab, uncomment these two lines to download:
# from google.colab import files
# files.download(str(ZIP_PATH))

## 12. Interpretation, limits, and next experiments

### What this experiment demonstrates

The Transformer never receives an explicit grammar or a database of detective facts. It learns statistical regularities from repeated next-token prediction: names, punctuation, dialogue patterns, sentence rhythm, and associations among clues, rooms, letters, doors, London, Holmes, and Watson.

### What it does **not** demonstrate

- It does not reason like Sherlock Holmes.
- It does not verify facts or preserve story continuity reliably.
- It does not understand sentences in the human sense.
- A tiny corpus can cause memorization and stylistic imitation.
- Word-level `<unk>` tokens limit its handling of unfamiliar names and vocabulary.

### Useful extensions

1. Replace the word tokenizer with byte-pair encoding.
2. Hold out an entire story or book for a stricter test.
3. Add beam search and compare it with stochastic sampling.
4. Compare the from-scratch model with a fine-tuned pretrained model.
5. Implement BERT-style binary next-sentence classification as a separate experiment.
6. Add an audit manifest recording URLs, download hashes, configuration, runtime, and random seeds.

---

## References

- Arthur Conan Doyle texts: [Project Gutenberg](https://www.gutenberg.org/)
- Vaswani et al. (2017), [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- PyTorch, [TransformerEncoder documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html)

**Pedagogical conclusion:** the model predicts the next sentence only because thousands of smaller next-token decisions are connected through masked self-attention. The sentence is the visible result; token prediction is the engine.